# Appendix 10.2: Tool Use

**Technique:** Tool use (function calling)

Give Claude typed **tools** (functions you define). When Claude decides to use one it returns a `tool_use` block instead of plain text. You execute the function locally, send back a `tool_result` block, and repeat until `stop_reason !== "tool_use"`. The loop is just plain JavaScript, so you stay in control of every execution.

**Contents**
* [Lesson: Tool Definitions, tool_choice, and the Agentic Loop](#lesson)
* [Worked Example: Calculator](#example)
* [Exercises](#exercises)
* Common Mistakes, Stretch, and Reflect
* [Example Playground](#playground)

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Lesson: Tool Definitions, tool_choice, and the Agentic Loop {#lesson}

### Tool definition shape

Each tool is a plain object with three required fields:

```js
{
  name: "myTool",                          // snake_case identifier Claude uses in tool_use blocks
  description: "What this tool does …",   // critical: Claude reads this to decide when to call it
  input_schema: {                          // JSON Schema for the arguments Claude must supply
    type: "object",
    properties: {
      argName: { type: "string", description: "…" },
    },
    required: ["argName"],
  },
}
```

Pass an array of these as `tools` to `client.messages.create`.

### tool_choice

By default Claude decides whether to use a tool. You can override this:

* `{ type: "auto" }`: Claude decides (default)
* `{ type: "any" }`: Claude must use at least one tool
* `{ type: "tool", name: "myTool" }`: Claude must call this specific tool

### The agentic loop

```
1. Send messages + tools → Claude
2. If stop_reason === "tool_use":
     a. Append assistant content to messages
     b. For each tool_use block: run the tool locally
     c. Append a user message with tool_result blocks
     d. Send again → go to 2
3. stop_reason !== "tool_use" → done; read the text response
```

Key invariants:
* **Always append the full `response.content` array** (not just the text block) as the next assistant message, which preserves the `tool_use` blocks in context.
* **Match `tool_use_id`** in each `tool_result` to the exact `id` from the `tool_use` block.
* **Never break the loop early**, since Claude may make multiple successive tool calls.

In [ ]:
// Worked example: a calculator tool with a manual agentic loop.
// NOTE: Function() is used only for this sandboxed demo; never eval untrusted input in production.

const tools = [{
  name: "calculator",
  description: "Evaluate a basic arithmetic expression and return the numeric result.",
  input_schema: {
    type: "object",
    properties: { expression: { type: "string", description: "e.g. '12 * (3 + 4)'" } },
    required: ["expression"],
  },
}];

function runTool(name, input) {
  if (name === "calculator") {
    // NOTE: Function() is used only for this sandboxed demo; never eval untrusted input in production.
    return String(Function(`"use strict"; return (${input.expression});`)());
  }
  return "unknown tool";
}

let messages = [{ role: "user", content: "What is 12 * (3 + 4)? Use the calculator tool." }];
let response = await client.messages.create({ model: MODEL, max_tokens: 1024, temperature: 0, tools, messages });

while (response.stop_reason === "tool_use") {
  messages.push({ role: "assistant", content: response.content });
  const results = [];
  for (const block of response.content) {
    if (block.type === "tool_use") {
      results.push({ type: "tool_result", tool_use_id: block.id, content: runTool(block.name, block.input) });
    }
  }
  messages.push({ role: "user", content: results });
  response = await client.messages.create({ model: MODEL, max_tokens: 1024, temperature: 0, tools, messages });
}

console.log(response.content.find((b) => b.type === "text").text);

## Exercises {#exercises}

### Exercise 10.2.1: Query a mock service

**Scenario**: Claude needs live operational data it doesn't have. You provide a `getServiceStatus(service)` tool backed by a mock `STATUS` object defined in the cell. Claude calls the tool to look up the current status, then answers the question using the returned value.

**Task**: fill in the `tools` array by defining a `getServiceStatus` tool whose `input_schema` accepts a `service` string. The `runTool` function and the agentic loop are already written for you.

**Success criteria**: after running the loop, the final assistant text contains the mock status value `"degraded"` for the billing service.

In [ ]:
import { includesAny, grade } from "../lib/grading.js";

const STATUS = { billing: "degraded", auth: "operational", search: "operational" };

const tools = [{
  name: "getServiceStatus",
  description: "Get the current operational status of a named service. Returns a status string such as 'operational' or 'degraded'.",
  input_schema: {
    type: "object",
    properties: {
      service: { type: "string", description: "The service name to look up, e.g. 'billing', 'auth', or 'search'." },
    },
    required: ["service"],
  },
}];

function runTool(name, input) {
  if (name === "getServiceStatus") return STATUS[input.service] ?? "unknown";
  return "unknown tool";
}

let messages = [{ role: "user", content: "Is the billing service up? Use the tool, then tell me its status." }];
let response = await client.messages.create({ model: MODEL, max_tokens: 1024, temperature: 0, tools, messages });
while (response.stop_reason === "tool_use") {
  messages.push({ role: "assistant", content: response.content });
  const results = [];
  for (const block of response.content) {
    if (block.type === "tool_use") results.push({ type: "tool_result", tool_use_id: block.id, content: runTool(block.name, block.input) });
  }
  messages.push({ role: "user", content: results });
  response = await client.messages.create({ model: MODEL, max_tokens: 1024, temperature: 0, tools, messages });
}
const finalText = (response.content.find((b) => b.type === "text") || {}).text || "";
const gradeExercise = (text) => includesAny(text, ["degraded"]);
grade(finalText, gradeExercise(finalText));

In [ ]:
import { exercise_10_2_1_hint } from "../hints.js";
console.log(exercise_10_2_1_hint);

## Common Mistakes, Stretch, and Reflect {#common}

### Common mistakes

**Forgetting to append the assistant `content` before the `tool_result`**: the API requires the full assistant turn (which includes the `tool_use` block) to appear in the message history immediately before the corresponding `tool_result` user turn. Skipping this step causes a validation error.

**Mismatched `tool_use_id`**: each `tool_result` block must carry the exact `id` from the `tool_use` block that triggered it. Copy it verbatim. Don't reconstruct or guess it.

**Not looping**: Claude may decide to call multiple tools in sequence or call a tool more than once before giving a final text answer. Calling the API only once misses every turn after the first. Always loop on `stop_reason === "tool_use"`.

### Stretch: the SDK tool runner

Writing the agentic loop by hand is instructive but repetitive. The Anthropic SDK ships a more convenient helper that automates it:

```js
// Stretch: the SDK's tool runner automates the whole loop for you.
// import { betaZodTool } from "npm:@anthropic-ai/sdk/helpers/beta/zod";
// import { z } from "npm:zod";
//
// const calculator = betaZodTool({
//   name: "calculator",
//   description: "Evaluate a basic arithmetic expression.",
//   inputSchema: z.object({ expression: z.string() }),
//   run: ({ expression }) => String(evaluate(expression)),
// });
//
// const runner = client.beta.messages.toolRunner({
//   model: MODEL,
//   max_tokens: 1024,
//   tools: [calculator],
//   messages: [{ role: "user", content: "What is 12 * (3 + 4)?" }],
// });
// const finalMessage = await runner; // the runner drives the tool loop until Claude is done
```

`betaZodTool` + `client.beta.messages.toolRunner` handle the loop, tool dispatch, and result injection for you. Use the manual loop while learning; reach for the tool runner in production code.

### Reflect

**When do you promote a bash command to a dedicated typed tool?**

A bash command is convenient for occasional ops, but a typed tool is worth the overhead when:

* **The operation needs validation**: a typed `input_schema` catches bad arguments before they hit your infrastructure.
* **Claude needs to select from multiple capabilities**: named tools let Claude reason about which action to take, rather than constructing a freeform shell string.
* **You want auditability**: typed tool calls are structured and loggable; freeform shell strings are not.
* **The operation is dangerous**: wrapping a destructive action in a tool lets you add confirmation logic or rate limiting between the tool call and the actual execution.

## Example Playground {#playground}

Use the cell below to experiment with your own tool definitions. Try adding a second tool (e.g., `getCurrentTime` or `lookupUser`) and watch how Claude decides which to call based on the question.

In [ ]:
// Build your own tool use here.
// Suggested ideas:
//   * A unit converter tool (input: value + from_unit + to_unit)
//   * A mock database lookup tool (input: table + id)
//   * A string formatter tool (input: text + format e.g. "upper" | "snake" | "kebab")

const myTools = [{
  name: "toUpperCase",
  description: "Convert a string to upper case.",
  input_schema: {
    type: "object",
    properties: { text: { type: "string", description: "The string to convert." } },
    required: ["text"],
  },
}];

function myRunTool(name, input) {
  if (name === "toUpperCase") return input.text.toUpperCase();
  return "unknown tool";
}

let myMessages = [{ role: "user", content: "Convert the phrase 'hello world' to upper case using the tool." }];
let myResponse = await client.messages.create({ model: MODEL, max_tokens: 1024, temperature: 0, tools: myTools, messages: myMessages });

while (myResponse.stop_reason === "tool_use") {
  myMessages.push({ role: "assistant", content: myResponse.content });
  const myResults = [];
  for (const block of myResponse.content) {
    if (block.type === "tool_use") myResults.push({ type: "tool_result", tool_use_id: block.id, content: myRunTool(block.name, block.input) });
  }
  myMessages.push({ role: "user", content: myResults });
  myResponse = await client.messages.create({ model: MODEL, max_tokens: 1024, temperature: 0, tools: myTools, messages: myMessages });
}

console.log(myResponse.content.find((b) => b.type === "text").text);